# Neural ODE / Continious Normalizing on MNIST-Dataset

In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchinfo import summary
from torchdiffeq import odeint_adjoint, odeint
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
from umap import UMAP
import re


# if nvidia gpu available use it
device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("Nb of Devices: ", torch.cuda.device_count())
    print("Device:",[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
    print("Compute capability:", torch.cuda.get_device_capability(0))
    print("Supported archs:", torch.cuda.get_arch_list())

## 1. Loading MNIST-Dataset

In [ ]:
# download test and trainset of mnist via pytorchvision
# dequantization and logit transform to make pixel value bins continoues
lambd = 1e-6
def preprocess(x):
    x = x * 255.0                          # <-- DAS fehlte: zurück auf {0,...,255}
    x = (x + torch.rand_like(x)) / 256.0
    x = lambd + (1 - 2*lambd) * x
    return torch.log(x) - torch.log1p(-x)

# define a transformation from PIL to tensor with normalized pixels
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(preprocess)
])

# download mnist dataset
train_set = datasets.MNIST(root="./data", train=True, download=True, transform=transform)

# get images
x = train_set.data.float() / 255.0
x = lambd + (1 - 2*lambd) * x
images = torch.log(x) - torch.log1p(-x)

labels = train_set.targets

print()
print(train_set)
print()
print("shape images: ", images.shape)

In [ ]:
# plot example images
fig, ax = plt.subplots(3, 5, figsize=(10, 5))
ax = ax.flatten()
for i in range(0, 60, 4):
    ax[int(i/4)].imshow(images[i])
fig.tight_layout()

In [ ]:
# reduce multidimensional data distribution onto 2D plain with UMAP
reducer = UMAP(n_neighbors=5, n_components=2, min_dist=0.4, metric="euclidean")

# fit reducer onto data
X = images.reshape(len(train_set), -1)
reducer.fit(X)

# make embedding
embedding_train_set = reducer.transform(X).T

In [ ]:
# plot embedding
print(embedding_train_set.shape)

fig, ax = plt.subplots(figsize=(8, 8))
scatter = ax.scatter(embedding_train_set[0], embedding_train_set[1], s=0.1, alpha=0.6, c=labels, cmap="tab10")
legend = ax.legend(*scatter.legend_elements(), title="Digit", loc="best")
ax.add_artist(legend)

ax.set_title("UMAP Projection of MNIST")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
plt.tight_layout()
plt.show()

## 2. Define Model

In [ ]:
class SingleFlow(nn.Module):
    def __init__(self, input_channel):
        super().__init__()
        # Flow layers
        self.conv1 = nn.Conv2d(in_channels=input_channel+1, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(in_channels=64+1, out_channels=64, kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64+1, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.conv4 = nn.Conv2d(in_channels=128+1, out_channels=128, kernel_size=3, stride=2, padding=1)
        self.transconv1 = nn.ConvTranspose2d(in_channels=128+1, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.transconv2 = nn.ConvTranspose2d(in_channels=128+1, out_channels=64, kernel_size=4, stride=2, padding=1)
        self.transconv3 = nn.ConvTranspose2d(in_channels=64+1, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.transconv4 = nn.ConvTranspose2d(in_channels=64+1, out_channels=input_channel, kernel_size=4, stride=2, padding=1)
        self.activation = nn.Softplus()

    def _tc(self, y, t):
        # concatenate t as extra channel
        return torch.cat([y, t.view(-1,1,1,1).expand(y.shape[0],1,y.shape[2],y.shape[3])], dim=1)

    def forward(self, t, y):
        # CNN
        y = self.conv1(self._tc(y, t))
        y = self.activation(y)
        y = self.conv2(self._tc(y, t))
        y = self.activation(y)
        y = self.conv3(self._tc(y, t))
        y = self.activation(y)
        y = self.conv4(self._tc(y, t))
        y = self.activation(y)
        y = self.transconv1(self._tc(y, t))
        y = self.activation(y)
        y = self.transconv2(self._tc(y, t))
        y = self.activation(y)
        y = self.transconv3(self._tc(y, t))
        y = self.activation(y)
        y = self.transconv4(self._tc(y, t))
        
        return y


class RNODE(nn.Module):
    def __init__(self, channels=1):
        super().__init__()
        self.channels=channels

        # track NFE
        self.nfe = 0

        # RNODE Net
        self.net = SingleFlow(self.channels)

    def forward(self, t, x_t):
        self.nfe += 1
                
        # calculate divergence/trace
        with torch.enable_grad():
            y = x_t.requires_grad_(True)
            dy_dt = self.net(t, y)

        return dy_dt


## 3. Training Model

In [ ]:
# train step function for Neural ODEs wit flow matching
def make_train_step(model, optimizer, grad_clipping, device):

    # training function that gets created
    def CFM_step(x_1, sigma_min=0.001, verbose=False):
        optimizer.zero_grad()
        model.train()

        # sample normal and uniform
        batch_size = x_1.shape[0]
        x_0 = torch.randn_like(x_1) 
        t = torch.rand((batch_size, 1, 1, 1), device=device)

        # Model eval
        x_t = (1-(1-sigma_min)*t)*x_0 + t*x_1
        model_eval = model(t, x_t)
        
        ## CFM loss
        flow = x_1 - (1-sigma_min)*x_0
        loss = torch.mean((model_eval - flow) ** 2) 
        
        # calculate gradient
        loss.backward()
        
        # verbose while training
        if verbose:
            print("=== Total Loss ===")
            print(f"{loss.mean().item():.7f}")
            print(f"NFE: {model.nfe}")
            print()
        
        # gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clipping)

        # optimization and scheduler
        optimizer.step()

        return loss.item(), model.nfe
    
    return CFM_step

In [ ]:
# training parameters
epochs = 8
lr = 1e-4
batch_size = 512
gradient_clip = 8
weight_decay = 1e-4

# init model
model = RNODE().to(device)

# define an optimizer
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

# define lr scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.7)

In [ ]:
# Turn Datasets into Dataloader
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)

# create training function
training_step = make_train_step(model, optimizer, gradient_clip, device=device)

# Training Loop
train_history = []
for e in range(epochs):
    print(f"\n========== Epoch: {e+1} ===========")
    
    batch_loss = []
    batch_nfe = []
    for i, (image_batch, _) in enumerate(train_loader, 1):
        image_batch = image_batch.to(device)

        # verbose nur bei bestimmten Epochen
        verbose = (e+1) % 2 == 0 or e+1 == 1

        # perform a train step
        loss_b, nfe = training_step(image_batch, verbose=verbose)
        batch_loss.append(loss_b)
        batch_nfe.append(nfe)

        if i % 10 == 0:
            with open("train_history.txt", "a") as f:
                f.write(f"batch: {i}, loss: {loss_b:.7f}\n")
                f.write(f"nfe: {torch.ceil(nfe)}\n")

    
    # calculate loss per epoch
    loss_e = np.array(batch_loss).mean()
    nfe_e = np.array(batch_nfe).mean()

    with open("train_history.txt", "a") as f:
        f.write(f"epoch: {e+1}\n")
        f.write(f"epoch loss: {loss_e:.7f}\n")
        f.write(f"epoch nfe: {torch.ceil(nfe_e)}\n")
        f.write(f"Learning Rate: {optimizer.param_groups[0]['lr']:.2e}\n")

    # learning rate scheduler
    scheduler.step(loss_e)
    
    # training statistics
    train_history.append(loss_e)
    print(f"Loss: {loss_e:.7f}, Learning Rate: {optimizer.param_groups[0]['lr']:.2e}")

    # Model Checkpoint
    if (e+1) % 5 == 0:
        torch.save(model.state_dict(), f"flowmatching_checkpoint_epoch{e+1}.pt")
        print(f"Checkpoint saved (epoch {e+1})")

## 4. Evaluation

In [ ]:
def load_train_history(filepath):
    train_history = []
    with open(filepath, "r") as f:
        content = f.read()
    
    parts = content.split("epoch loss:")
    for part in parts[1:]:
        loss = float(part.split()[0])        
        train_history.append(loss)
    
    return train_history

In [ ]:
# load model and train history
#model.load_state_dict(torch.load("/kaggle/input/models/mikematician/reg3/pytorch/default/1/cnf_checkpoint_epoch8.pt", map_location=device))
#train_history = load_train_history("/kaggle/input/models/mikematician/reg3/pytorch/default/1/train_history.txt")

In [ ]:
# Train Loss
plt.figure()
plt.plot(range(1, len(train_history)+1), train_history)
plt.title("Train History")
plt.ylabel("Loss")
plt.xlabel("Epochs")
plt.show()

In [ ]:
# Sample Function
@torch.no_grad()
def sample(model, num_samples, t_0, t_end, ode_solver, device, rtol, atol, shape=(1, 28, 28)):
    model.eval()
    y_0 = torch.randn(num_samples, *shape, device=device)
    # set epsilon for hutchinson trace estimate
    #model.sample_eps(y_0)
    y_1 = odeint(model, y_0, torch.tensor([t_0, t_end], dtype=torch.float32, device=device), rtol=rtol, atol=atol, method=ode_solver)
    
    return y_1, y_0

In [ ]:
nb_samples = 100
t_0 = 0
t_1 = 1
ode_solver = "dopri5"
atol = 1e-5
rtol = 1e-5

generated_images , gaussian_noise = sample(model, nb_samples, t_0, t_1, ode_solver, device, rtol=rtol, atol=atol)

# convert to cpu
generated_images_cpu = generated_images.cpu()
gaussian_noise_cpu = gaussian_noise.cpu()

In [ ]:
# plot example images
nb_plots = 10

fig, ax = plt.subplots(2, nb_plots, figsize=(2*nb_plots+2, 4))
ax = ax.flatten()
for i in range(0, nb_plots):
    ax[i].imshow(generated_images_cpu[i].squeeze(), cmap="gray")
    ax[i].set_title(f"generated image {i+1}")
    ax[i+nb_plots].imshow(gaussian_noise_cpu[i].squeeze(), cmap="gray")
    ax[i+nb_plots].set_title(f"gaussian noise {i+1}")
fig.tight_layout()


In [ ]:
# Umap exploration
# nb of real samples
n_real = 5000

# real flat samples
X_real   = images[:n_real].reshape(n_real, -1).numpy().astype("float32")
L_real   = labels[:n_real].numpy()

# faltten generated data
X_gen = generated_images_cpu.reshape(len(generated_images_cpu), -1).numpy()

# project into learned umap projection
emb_real = reducer.transform(X_real).T
emb_gen  = reducer.transform(X_gen).T  

# plot
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# all data
ax = axes[0]
sc = ax.scatter(emb_real[0], emb_real[1], c=L_real, cmap="tab10", s=1.5, alpha=0.5)
legend = ax.legend(*sc.legend_elements(), title="Digit", loc="best", markerscale=3)
ax.add_artist(legend)
ax.scatter(emb_gen[0], emb_gen[1], color="red", s=60, zorder=5, marker="*", label="generated")
ax.legend(loc="upper right")
ax.set_title("UMAP: Real Samples + Generated (★)")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")

# grey overlay
ax = axes[1]
ax.scatter(emb_real[0], emb_real[1], color="lightgrey", s=1.5, alpha=0.4, label=f"real (n={n_real})")
ax.scatter(emb_gen[0], emb_gen[1], color="red", s=60, zorder=5, marker="*", label=f"generated (n={len(X_gen)})")
ax.legend(markerscale=3)
ax.set_title("UMAP: Overlay Real vs. Generated")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")

plt.tight_layout()
plt.show()

In [ ]:
print("gen std over all samples (per pixel, mean):", X_gen.std(axis=0).mean())
print("real std:", X_real.std(axis=0).mean())
print("sample[0] == sample[1]?", np.allclose(X_gen[0], X_gen[1]))

print("real  min/max/mean/std:", X_real.min(), X_real.max(), X_real.mean(), X_real.std())
print("gen   min/max/mean/std:", X_gen.min(),  X_gen.max(),  X_gen.mean(),  X_gen.std())